### GridWorld environment

In [688]:
UP, RIGHT, DOWN, LEFT = 'up', 'right', 'down', 'left'
ACTIONS = [UP, RIGHT, DOWN, LEFT]
DIRECTIONS = {UP: (-1, 0), RIGHT: (0, 1), DOWN: (1, 0), LEFT: (0, -1)}
ARROWS = {UP: "↑", RIGHT: "→", DOWN: "↓", LEFT: "←"}

In [689]:
class GridWorld:

    def __init__(
        self,
        rows: int,
        cols: int,
        step_reward: float,
        terminals: dict[tuple[int, int], float],
        walls: set[tuple[int, int]],
    ):
        self.rows = rows
        self.cols = cols
        self.step_reward = step_reward
        self.terminals = terminals
        self.walls = walls

    def step(self, cell: tuple[int, int], action: str):
        dr, dc = DIRECTIONS[action]
        next_cell = (cell[0] + dr, cell[1] + dc)
        outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
        on_wall = next_cell in self.walls
        if outside or on_wall:
            next_cell = cell
        if next_cell in self.terminals:
            reward = self.terminals[next_cell]
        else:
            reward = self.step_reward
        return next_cell, reward

    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return f"{self.terminals[cell]:+}"
        elif cell in self.walls:
            return "#"
        else:
            return "·"

    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print("".join([f"{v:>3} " for v in range(self.cols)]))
                if c == 0:
                    print(f"{r:>3} ", end="")
                print(f"{self.cell_repr(r, c):>3} ", end="")
            print()

    def __repr__(self):
        return f"""
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
)
"""

In [690]:
env = GridWorld(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)
env


Grid(
    rows=3,
    cols=4,
    step_reward=0,
    terminals={(0, 3): 1, (1, 3): -1},
    walls={(1, 1)},
)

In [691]:
env.render()

 r/c  0   1   2   3 
  0   ·   ·   ·  +1 
  1   ·   #   ·  -1 
  2   ·   ·   ·   · 


In [692]:
cell = (0, 0)
for _ in range(3):
    cell, reward = env.step(cell, RIGHT)
    print((cell, reward))

((0, 1), 0)
((0, 2), 0)
((0, 3), 1)


### value_iteration

In [693]:
def show_V(env: GridWorld, V: list[list[float]]):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print("".join([f"{v:>4} " for v in range(env.cols)]))
            if c == 0:
                print(f"{r:>4} ", end="")

            value = round(V[r][c], 2)
            print(f"{value:>4} ", end="")
        print()


def value_iteration(env: GridWorld, gamma=0.9, theta=1e-6):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    print('V init')
    show_V(env, V)
    print('-' * 25) 
    delta = float("inf")
    i = 0
    while delta >= theta:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                max_value = float("-inf")
                for action in ACTIONS:
                    next_cell, reward = env.step(cell, action)
                    action_value = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                    max_value = max(max_value, action_value)
                V[r][c] = max_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        print(f"iter {i}: delta={delta:.6f}")
        show_V(env, V)
        print("-" * 25)
        i += 1
    print(f"value_iteration converged in {i} iterations")
    return V

In [694]:
V = value_iteration(env)

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.

In [695]:
show_V(env, V)

  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2 0.66 0.73 0.81 0.73 


In [696]:
def read_policy(env: GridWorld, V: list[list[float]], gamma=0.9):
    policy = [[None for _ in range(env.cols)] for _ in range(env.rows)]
    for r in range(env.rows):
        for c in range(env.cols):
            cell = (r, c)
            if cell in env.walls or cell in env.terminals:
                continue

            max_value = float("-inf")
            for action in ACTIONS:
                next_cell, reward = env.step(cell, action)
                action_value = reward + gamma * V[next_cell[0]][next_cell[1]]
                if action_value > max_value:
                    policy[r][c] = action
                    max_value = action_value
    return policy


policy = read_policy(env, V)
policy

[['right', 'right', 'right', None],
 ['up', None, 'up', None],
 ['up', 'right', 'up', 'left']]

In [697]:
def render_policy(env: GridWorld, policy: list[list[str | None]]):
    for r in range(env.rows):
        for c in range(env.cols):
            if r == 0 and c == 0:
                print(" r/c", end="")
                print("".join([f"{v:>3} " for v in range(env.cols)]))
            if c == 0:
                print(f"{r:>3} ", end="")

            cell = (r, c)
            if cell in env.walls:
                value = '#'
            elif cell in env.terminals:
                value = f"{env.terminals[cell]:+}"
            elif policy[r][c] is not None:
                value = ARROWS[policy[r][c]]
            else:
                raise ValueError(f"no action for cell {cell}")

            print(f"{value:>3} ", end="")
        print()


render_policy(env, policy)

 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 


### read policy during value_iteration 

we can also read policy during value_iteration, no need to do it with returned V

In [698]:
def value_iteration(env: GridWorld, gamma=0.9, theta=1e-6):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    print('V init')
    show_V(env, V)
    print('-' * 25)  
    policy = [[None for _ in range(env.cols)] for _ in range(env.rows)]
    delta = float("inf")
    i = 0
    while delta >= theta:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                max_value = float("-inf")
                for action in ACTIONS:
                    next_cell, reward = env.step(cell, action)
                    action_value = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                    if action_value > max_value:
                        max_value = action_value
                        policy[r][c] = action
                V[r][c] = max_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        print(f"iter {i}: delta={delta:.6f}")
        show_V(env, V)
        print("-" * 25)
        i += 1
    print('policy:')
    render_policy(env, policy)
    print("-" * 25)
    print(f"value_iteration converged in {i} iterations")
    return policy, V

In [699]:
policy, V = value_iteration(env)

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.

### policy_evaluation behaves same as value_iteration under optimal policy

In [700]:
def policy_evaluation(
    env: GridWorld,
    policy: list[list[str | None]],
    gamma=0.9,
    theta=1e-6,
    verbose=True
):
    V = [[0.0 for _ in range(env.cols)] for _ in range(env.rows)]
    if verbose:
        print('V init')
        show_V(env, V)
        print('-' * 25)  
    delta = float("inf")
    i = 0
    while delta >= theta:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                action = policy[r][c]
                next_cell, reward = env.step(cell, action)
                V[r][c] = reward + gamma * V_old[next_cell[0]][next_cell[1]]
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        if verbose:
            print(f"iter {i}: delta={delta:.6f}")
            show_V(env, V)
            print("-" * 25)
        i += 1
    print(f"policy_evaluation converged in {i} iterations")
    return V

In [701]:
V = policy_evaluation(env, policy)

V init
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 0: delta=1.000000
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 1: delta=0.900000
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0  0.0  0.0 
-------------------------
iter 2: delta=0.810000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81  0.0 
-------------------------
iter 3: delta=0.729000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
iter 4: delta=0.656100
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.0 
   2 0.66 0.73 0.81 0.73 
-------------------------
iter 5: delta=0.000000
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.73  0.0  0.9  0.

### policy_iteration

In [702]:
def policy_iteration(env: GridWorld, gamma=0.9, theta=1e-6, pe_verbose=False):
    policy = [[UP for _ in range(env.cols)] for _ in range(env.rows)]
    changed = None
    i = 0
    while changed != 0:
        print(f'iter {i}:')
        V = policy_evaluation(env, policy, gamma, theta, verbose=pe_verbose)
        new_policy = read_policy(env, V, gamma)
        changed = 0
        for r in range(env.rows):
            for c in range(env.cols):
                cell = (r, c)
                if cell in env.walls or cell in env.terminals:
                    continue
                if new_policy[r][c] != policy[r][c]:
                    changed += 1
        print(f'V:')
        show_V(env, V)
        print('-' * 25)
        print(f'actions changed = {changed}')
        render_policy(env, new_policy)
        print('-' * 5)
        policy = new_policy
        i += 1
    print(f'policy_iteration converged in {i} iterations')
    return policy, V

In [703]:
policy, V = policy_iteration(env)

iter 0:
policy_evaluation converged in 2 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  0.0  0.0 
   1  0.0  0.0  0.0  0.0 
   2  0.0  0.0  0.0 -1.0 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   ↑   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   ↑   ↑   ← 
-----
iter 1:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.0  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0  0.0 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   ↑   →   →  +1 
  1   ↑   #   ↑  -1 
  2   ↑   →   ↑   ← 
-----
iter 2:
policy_evaluation converged in 5 iterations
V:
  r/c   0    1    2    3 
   0  0.0  0.9  1.0  0.0 
   1  0.0  0.0  0.9  0.0 
   2  0.0 0.73 0.81 0.73 
-------------------------
actions changed = 2
 r/c  0   1   2   3 
  0   →   →   →  +1 
  1   ↑   #   ↑  -1 
  2   →   →   ↑   ← 
-----
iter 3:
policy_evaluation converged in 6 iterations
V:
  r/c   0    1    2    3 
   0 0.81  0.9  1.0  0.0 
   1 0.7